<a href="https://colab.research.google.com/github/akashgardas/freight-rail-delay-prediction/blob/preprocessing/notebooks/preprocessing/Data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = "/content/drive/MyDrive/CVR College/Mini Project/data/"

In [ ]:
# Core tables only — skip WAGON_POSITION (1M+ rows, not needed yet)
gare        = pd.read_csv(DATA_PATH + "STATION.csv",        encoding="latin-1")
train       = pd.read_csv(DATA_PATH + "TRAIN.csv",          encoding="latin-1")
train_lot   = pd.read_csv(DATA_PATH + "TRAIN_LOT.csv",      encoding="latin-1")
train_jalon = pd.read_csv(DATA_PATH + "TRAIN_JALON.csv",    encoding="latin-1")
train_wagon = pd.read_csv(DATA_PATH + "TRAIN_WAGON.csv",    encoding="latin-1")
wagon_model = pd.read_csv(DATA_PATH + "WAGON.csv",          encoding="latin-1")

print("Shapes:")
for name, df in zip(
    ["gare","train","train_lot","train_jalon","train_wagon","wagon_model"],
    [ gare,  train,  train_lot,  train_jalon,  train_wagon,  wagon_model]
):
    print(f"  {name:15s}: {df.shape}")

/tmp/ipykernel_2218/2968890733.py:5: DtypeWarning: Columns (19,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  train_jalon = pd.read_csv(DATA_PATH + "TRAIN_JALON.csv",    encoding="latin-1")


Shapes:
  gare           : (7174, 19)
  train          : (14215, 19)
  train_lot      : (14692, 45)
  train_jalon    : (72531, 27)
  train_wagon    : (226998, 10)
  wagon_model    : (14, 43)


/tmp/ipykernel_2218/2968890733.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_wagon = pd.read_csv(DATA_PATH + "TRAIN_WAGON.csv",    encoding="latin-1")


In [ ]:
def check_df(df):
  print(df.shape)
  print('\nNull values:')
  print(df.isnull().sum())
  print('\nHEAD')
  return df.head()

## TRAIN_JALON

In [ ]:
train_jalon.columns

Index(['IDTRAIN_JALON', 'Jalon_num', 'Lieu', 'DHR_Arrivee', 'DHT_Arrivee',
       'DHT_Depart', 'DHR_Depart', 'IDTRAIN', 'IDGARE', 'Code_pays',
       'Latitude', 'Longitude', 'Jalon_passage', 'H_Depart_ecart',
       'H_Arrivee_ecart', 'IDUtilisateur_creation', 'DateH_creation',
       'IDUtilisateur_maj', 'DateH_maj', 'Commentaire', 'Distance_origine',
       'H_Arrivee_ecart1', 'H_Arrivee_ecart2', 'H_Depart_ecart1',
       'H_Depart_ecart2', 'DHO_Depart', 'DHO_Arrivee'],
      dtype='object')

In [ ]:
check_df(train_jalon[['H_Arrivee_ecart', 'H_Depart_ecart', 'Latitude', 'Longitude']])

(72531, 4)

Null values:
H_Arrivee_ecart    0
H_Depart_ecart     0
Latitude           0
Longitude          0
dtype: int64

HEAD


,H_Arrivee_ecart,H_Depart_ecart,Latitude,Longitude
0,0,60,0.00,0.00
1,60,0,0.00,0.00
2,60,60,0.00,0.00
3,60,60,0.00,0.00
4,60,60,0.00,0.00


## TRAIN_LOT

In [ ]:
train_lot.columns

Index(['IDTRAIN_LOT', 'IDTRAIN', 'Lot_num', 'DHR_Arrivee', 'DHT_Arrivee',
       'Nom_lot', 'Commentaire', 'Distance', 'Poids_CO2',
       'IDSOCIETE_Expediteur', 'IDSOCIETE_Destinataire', 'IDJALON_Origine',
       'IDJALON_Destination', 'Surbooking', 'Objectif_CA', 'Objectif_TEU',
       'DHT_Depart', 'DHR_Depart', 'H_Depart_ecart', 'H_Arrivee_ecart',
       'IDUtilisateur_creation', 'DateH_creation', 'IDUtilisateur_maj',
       'DateH_maj', 'Ref_Expediteur', 'Ref_Destinataire', 'Train_Position',
       'Incoterm', 'Num_LDV', 'Objectif_T', 'Objectif_UTI', 'Nbre_TEU',
       'Capacite_TEU', 'Nbre_UTI', 'Poids_total', 'Status', 'Longueur_totale',
       'Voie_categorie', 'Voie_vitesse', 'Motif_suppression',
       'DateH_suppression', 'IDSOCIETE_Client', 'DateH_info_fournisseur',
       'Responsable_annulation', 'Top_reservation_interdite'],
      dtype='object')

In [ ]:
check_df(train_lot[['Nbre_TEU', 'Longueur_totale', 'Distance', 'Poids_total']])

(14692, 4)

Null values:
Nbre_TEU           43
Longueur_totale    67
Distance            0
Poids_total        43
dtype: int64

HEAD


,Nbre_TEU,Longueur_totale,Distance,Poids_total
0,0.00,390.00,99,353600.00
1,NaN,NaN,99,NaN
2,NaN,NaN,99,NaN
3,NaN,NaN,99,NaN
4,NaN,NaN,99,NaN


## Wagon Count

In [ ]:
# Count wagons per train lot — used later for weight_per_wagon (x7)
wagon_count = (
    train_wagon
    .groupby("IDTRAIN_LOT")["IDTRAIN_WAGON"]
    .count()
    .reset_index()
    .rename(columns={"IDTRAIN_WAGON": "wagon_count"})
)

print(wagon_count["wagon_count"].describe())

count   11775.00
mean       19.28
std         6.41
min         1.00
25%        17.00
50%        21.00
75%        21.00
max        36.00
Name: wagon_count, dtype: float64


In [ ]:
wagon_count.head()

,IDTRAIN_LOT,wagon_count
0,7138,22
1,7139,22
2,7140,22
3,7141,22
4,7142,22


## Merging datasets

In [ ]:
df_merged = train_jalon

In [ ]:
df_merged.shape

(72531, 27)

In [ ]:
# Merge with train lot
df_merged = df_merged.merge(
    train_lot[['IDTRAIN', 'IDTRAIN_LOT', 'Distance', 'Poids_total', 'Longueur_totale', 'Nbre_TEU']],
    on='IDTRAIN',
    how='left'
)

In [ ]:
df_merged.shape

(75855, 32)

In [ ]:
# Merge with wagon count
df_merged = df_merged.merge(
    wagon_count,
    on='IDTRAIN_LOT',
    how='left'
)

In [ ]:
df_merged.shape

(75855, 33)

## Extracting features

In [ ]:
df = pd.DataFrame()

In [ ]:
# y - target feature
df['y_arrival_delay_time'] = df_merged.H_Arrivee_ecart
check_df(df)

(75855, 1)

Null values:
y_arrival_delay_time    0
dtype: int64

HEAD


,y_arrival_delay_time
0,0
1,60
2,60
3,60
4,60


In [ ]:
# x1 - teu_count
df['x1_teu_count'] = df_merged.Nbre_TEU
check_df(df)

(75855, 2)

Null values:
y_arrival_delay_time      0
x1_teu_count            329
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count
0,0,0.00
1,60,0.00
2,60,0.00
3,60,0.00
4,60,0.00


In [ ]:
# x2 — train_length [m]
df['x2_train_length'] = df_merged.Longueur_totale
check_df(df)

(75855, 3)

Null values:
y_arrival_delay_time      0
x1_teu_count            329
x2_train_length         514
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count,x2_train_length
0,0,0.00,390.00
1,60,0.00,390.00
2,60,0.00,390.00
3,60,0.00,390.00
4,60,0.00,390.00


In [ ]:
# total_distance_trip [km]
df['x3_total_distance_trip'] = df_merged.Distance
check_df(df)

(75855, 4)

Null values:
y_arrival_delay_time        0
x1_teu_count              329
x2_train_length           514
x3_total_distance_trip      0
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count,x2_train_length,x3_total_distance_trip
0,0,0.00,390.00,99
1,60,0.00,390.00,99
2,60,0.00,390.00,99
3,60,0.00,390.00,99
4,60,0.00,390.00,99


In [ ]:
# x4 - departure_delay_time [min]
df['x4_departure_delay_time'] = df_merged.H_Depart_ecart
check_df(df)

(75855, 5)

Null values:
y_arrival_delay_time         0
x1_teu_count               329
x2_train_length            514
x3_total_distance_trip       0
x4_departure_delay_time      0
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count,x2_train_length,x3_total_distance_trip,x4_departure_delay_time
0,0,0.00,390.00,99,60
1,60,0.00,390.00,99,0
2,60,0.00,390.00,99,60
3,60,0.00,390.00,99,60
4,60,0.00,390.00,99,60


In [ ]:
# x5 - distance_between_stations

In [ ]:
# x6 — weight_per_length [t/m]
df['x6_weight_per_length'] = df_merged.Poids_total / df_merged.Longueur_totale
check_df(df)

(75855, 6)

Null values:
y_arrival_delay_time          0
x1_teu_count                329
x2_train_length             514
x3_total_distance_trip        0
x4_departure_delay_time       0
x6_weight_per_length       1143
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count,x2_train_length,x3_total_distance_trip,x4_departure_delay_time,x6_weight_per_length
0,0,0.00,390.00,99,60,906.67
1,60,0.00,390.00,99,0,906.67
2,60,0.00,390.00,99,60,906.67
3,60,0.00,390.00,99,60,906.67
4,60,0.00,390.00,99,60,906.67


In [ ]:
# x7 — weight_per_wagon [t/wagon]
df['x7_weight_per_wagon'] = df_merged.Poids_total / df_merged.wagon_count
check_df(df)

(75855, 7)

Null values:
y_arrival_delay_time           0
x1_teu_count                 329
x2_train_length              514
x3_total_distance_trip         0
x4_departure_delay_time        0
x6_weight_per_length        1143
x7_weight_per_wagon        13920
dtype: int64

HEAD


,y_arrival_delay_time,x1_teu_count,x2_train_length,x3_total_distance_trip,x4_departure_delay_time,x6_weight_per_length,x7_weight_per_wagon
0,0,0.00,390.00,99,60,906.67,NaN
1,60,0.00,390.00,99,0,906.67,NaN
2,60,0.00,390.00,99,60,906.67,NaN
3,60,0.00,390.00,99,60,906.67,NaN
4,60,0.00,390.00,99,60,906.67,NaN


## Save the dataset

In [ ]:
# save
df.to_csv('extracted_dataset.csv')